# Agent Failure Modes

Agents fail in characteristic, predictable ways. This notebook demonstrates the five most common failure modes using a realistic production scenario. Understanding these patterns is the first step to preventing them.

**Use case: Credit risk assessment for a private company**

Meridian Logistics Ltd, a private UK logistics company, has applied for a £2M trade credit line. The agent must assess their creditworthiness.

Private company financial data is inherently incomplete. Unlike public companies, there are no quarterly filings, no analyst coverage,
and no real-time market data. This creates natural information gaps that trigger every major failure mode — hallucination, infinite retry, goal drift, shallow reasoning, and knowledge mismatch.


We will see the five failures in agents:
1. **Hallucination loop**: agent invents data when tools return nothing useful
2. **Infinite retry**: agent retries a failing tool without changing strategy
3. **Goal drift**: agent pursues tangential information and loses sight of the goal
4. **Shallow reasoning**: agent stops after one positive signal without full analysis
5. **Knowledge mismatch**: agent assumes private company data works like public data


## Setup

In [1]:
import os
import google.genai as genai

client = genai.Client()

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


---
## The Task


In [2]:
TASK = (
    "Assess the credit risk for Meridian Logistics Ltd, a private logistics company "
    "based in Manchester, UK. They are applying for a GBP 2M trade credit line. "
    "Evaluate their financial health, payment history, and industry risk profile. "
    "Provide a credit recommendation: approve, approve with conditions, or decline."
)
print("Task:", TASK)


Task: Assess the credit risk for Meridian Logistics Ltd, a private logistics company based in Manchester, UK. They are applying for a GBP 2M trade credit line. Evaluate their financial health, payment history, and industry risk profile. Provide a credit recommendation: approve, approve with conditions, or decline.


---
## Tool Implementations

The tools below simulate a real credit assessment environment.
Some tools work (credit bureau, industry data). Others hit expected dead ends
(no public filings for a private company, no real-time market data).

**The dead ends are intentional.** A well-designed agent must recognise when
a tool won't yield useful data and change strategy rather than retry blindly.


In [3]:
# Tool implementations

def get_company_financials(company_name: str, report_type: str) -> str:
    # Attempts to retrieve financial statements.
    # Private companies have no public financials except via credit bureau.
    name = company_name.lower()
    rtype = report_type.lower()
    if "meridian" in name:
        if rtype in ("credit_bureau", "credit bureau", "credit_report"):
            return (
                "Meridian Logistics Ltd | Credit bureau report: "
                "Incorporated: 2009 | Employees est.: 180-220 | "
                "Registered capital: GBP 250K | "
                "Credit score: 62/100 (moderate risk) | "
                "Outstanding CCJs: 1 (GBP 18,500, 2022, settled) | "
                "Credit utilisation: 74% of current facilities | "
                "Longest credit relationship: 11 years (Barclays)"
            )
        if rtype in ("annual_report", "audited", "accounts", "p&l", "balance_sheet"):
            return (
                "DATA_UNAVAILABLE: Meridian Logistics Ltd is a private company. "
                "Audited accounts are not publicly available. "
                "Companies House filings show abbreviated accounts only (no P&L). "
                "Consider using report_type='credit_bureau' for available data."
            )
        return f"DATA_UNAVAILABLE: No financial data for report_type='{report_type}'"
    return f"No data found for '{company_name}'"


def get_payment_history(company_name: str) -> str:
    # Returns payment history from trade credit database.
    if "meridian" in company_name.lower():
        return (
            "Meridian Logistics Ltd | Payment history (36 months): "
            "On-time payments: 79% | Late (1-30 days): 17% | Late (30+ days): 4% | "
            "Average days late: 8 | Largest overdue incident: GBP 45K (Q3 2023, resolved in 47 days) | "
            "Current overdue: GBP 0 | Active trade credit relationships: 6"
        )
    return f"No payment history found for '{company_name}'"


def get_industry_risk_score(industry: str, region: str) -> str:
    # Returns industry-level risk metrics for a sector and geography.
    if "logistics" in industry.lower() or "freight" in industry.lower():
        return (
            f"Industry: {industry} | Region: {region} | "
            "Sector risk score: 58/100 (moderate-high) | "
            "Key risk factors: fuel cost volatility, driver shortage, e-commerce demand variability | "
            "Insolvency rate (UK logistics, 2023): 2.4% (vs 1.8% all industries) | "
            "Outlook: cautious -- margin compression due to energy costs continuing Q1 2025"
        )
    return f"No industry data for '{industry}' in '{region}'"


def get_public_filings(company_name: str, filing_type: str) -> str:
    # Searches public filing databases (Companies House, SEC, etc.)
    if "meridian" in company_name.lower():
        return (
            "DATA_UNAVAILABLE: Meridian Logistics Ltd files as a private limited company. "
            "Companies House records contain: certificate of incorporation, "
            "abbreviated balance sheets (no revenue/profit figures), director listings. "
            f"Filing type '{filing_type}' (P&L, revenue, EBITDA) is not publicly available "
            "for private UK companies unless they exceed the small company threshold."
        )
    return f"No public filings for '{company_name}'"


def search_news(company_name: str, topic: str) -> str:
    # Search for recent news about a company.
    key = f"{company_name} {topic}".lower()
    if "meridian" in key:
        if any(t in key for t in ("contract", "client", "win", "award", "customer")):
            return (
                "Meridian Logistics Ltd news (contracts): "
                "Renewed 3-year contract with Tesco distribution (announced June 2024). "
                "Lost Amazon last-mile contract (Q4 2023, awarded to competitor). "
                "New DHL partnership for cross-border (pilot phase)."
            )
        if any(t in key for t in ("financial", "revenue", "profit", "loss", "earnings")):
            return (
                "DATA_UNAVAILABLE: No public financial news for private company Meridian Logistics Ltd. "
                "No press releases on revenue or profitability found."
            )
        return "No significant news found for that topic."
    return f"No news found for '{company_name}'"


TOOLS = {
    "get_company_financials": get_company_financials,
    "get_payment_history":    get_payment_history,
    "get_industry_risk_score": get_industry_risk_score,
    "get_public_filings":     get_public_filings,
    "search_news":            search_news,
}

print("Tools loaded. Testing a dead-end case:")
print(get_company_financials("Meridian Logistics Ltd", "annual_report"))

Tools loaded. Testing a dead-end case:
DATA_UNAVAILABLE: Meridian Logistics Ltd is a private company. Audited accounts are not publicly available. Companies House filings show abbreviated accounts only (no P&L). Consider using report_type='credit_bureau' for available data.


---
## Failure Mode 1: Hallucination Loop

**What it is:** The agent receives `DATA_UNAVAILABLE` from a tool but invents a
plausible-sounding data point in its reasoning rather than acknowledging the gap.

**How to trigger it:** Ask for specific financial figures for a private company.
Without guard-rails, the agent may "reason" to a conclusion using invented revenue,
EBITDA margin, or debt figures it could not have retrieved.

**The underlying mechanism:** LLMs are trained to produce coherent, complete-sounding
responses. When context is absent, they fill the gap with statistically plausible text.
The agent loop amplifies this because the fabricated data enters the message history
and becomes "evidence" for subsequent reasoning.


In [ ]:
from google.genai import types

TOOL_DEFINITIONS = {
    "function_declarations": [

        {
            "name": "get_company_financials",
            "description": (
                "Retrieve financial statements for a company. "
                "report_type options: 'annual_report', 'credit_bureau', 'credit_report'. "
                "Note: private companies may only have credit_bureau data available."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "company_name": {
                        "type": "string",
                        "description": "Full company name."
                    },
                    "report_type": {
                        "type": "string",
                        "description": "Type of financial data: annual_report, credit_bureau, credit_report."
                    }
                },
                "required": ["company_name", "report_type"]
            }
        },

        {
            "name": "get_payment_history",
            "description": "Returns 36-month payment history from the trade credit database.",
            "parameters": {
                "type": "object",
                "properties": {
                    "company_name": {
                        "type": "string",
                        "description": "Full company name."
                    }
                },
                "required": ["company_name"]
            }
        },

        {
            "name": "get_industry_risk_score",
            "description": "Returns industry-level risk score and sector outlook for a given industry and region.",
            "parameters": {
                "type": "object",
                "properties": {
                    "industry": {
                        "type": "string",
                        "description": "Industry or sector name."
                    },
                    "region": {
                        "type": "string",
                        "description": "Geographic region, e.g. UK."
                    }
                },
                "required": ["industry", "region"]
            }
        },

        {
            "name": "get_public_filings",
            "description": "Search public filing databases for company financial disclosures.",
            "parameters": {
                "type": "object",
                "properties": {
                    "company_name": {
                        "type": "string",
                        "description": "Full company name."
                    },
                    "filing_type": {
                        "type": "string",
                        "description": "Type of filing sought, e.g. annual accounts, P&L, EBITDA."
                    }
                },
                "required": ["company_name", "filing_type"]
            }
        },

        {
            "name": "search_news",
            "description": "Search recent news about a company on a specific topic.",
            "parameters": {
                "type": "object",
                "properties": {
                    "company_name": {
                        "type": "string",
                        "description": "Full company name."
                    },
                    "topic": {
                        "type": "string",
                        "description": "Topic to search for, e.g. 'financial results' or 'contracts'."
                    }
                },
                "required": ["company_name", "topic"]
            }
        }

    ]
}

---
## Agent Loop with Failure Mode Instrumentation

We augment the standard agent loop with **instrumentation** that detects and
labels failure patterns as they occur. In a production system this would be
part of an observability layer.


In [ ]:
# a weak system prompt to increase failure surface
def model(msg):
    return client.models.generate_content(model='gemini-2.5-flash-lite',
                                              contents=msg,
                                              config = types.GenerateContentConfig(
                                                  system_instruction=(
                                                        "You are a credit risk analyst. Assess the creditworthiness of the company. "
                                                        "Use the tools to retrieve data and produce a recommendation."
                                                    ),
                                                  tools=[TOOL_DEFINITIONS])
                            )

print("Model ready with", len(TOOL_DEFINITIONS['function_declarations']), "tools.")

Model ready with 5 tools.


In [6]:
class InstrumentedAgentState:
    def __init__(self, goal: str, max_steps: int = 15):
        self.goal          = goal
        self.messages      = []
        self.step_count    = 0
        self.max_steps     = max_steps
        self.done          = False
        self.final_answer  = None
        self.failure_log   = []           # (step, failure_type, description)
        self.tool_call_log = []           # (step, tool_name, args)
        self.unavailable_hits = {}        # tool_name -> count
        self.core_tools_called = set()    # which core tools were called

    def log_failure(self, failure_type: str, description: str):
        self.failure_log.append((self.step_count, failure_type, description))
        print(f"  [!] FAILURE DETECTED: {failure_type} -- {description}")

In [7]:
CORE_TOOLS = {"get_company_financials", "get_payment_history", "get_industry_risk_score"}
DEAD_END_TOOLS = {"get_public_filings"}

In [10]:
def run_instrumented_agent(state: InstrumentedAgentState, tools_dict=None) -> str:

    if tools_dict is None:
        tools_dict = TOOLS

    state.messages.append({"role": "user", "parts": [{"text": state.goal}]})
    print("=" * 65)
    print(f"TASK: {state.goal[:100]}...")
    print("=" * 65)

    while not state.done:
        
        if state.step_count >= state.max_steps:
            state.log_failure("MAX_STEPS_EXCEEDED",
                f"Reached {state.max_steps} steps without resolution")
            state.final_answer = "Stopped: max steps exceeded."
            state.done = True
            break

        state.step_count += 1
        print(f"\n[Step {state.step_count}]")
        response = model(state.messages)
        parts    = response.candidates[0].content.parts

        tool_calls = [
                {"name": p.function_call.name, "args": dict(p.function_call.args)}
                for p in parts
                if getattr(p, "function_call", None) is not None
            ]


# FM: Shallow reasoning -- concluded without calling all core tools
        if not tool_calls:
            missing = CORE_TOOLS - state.core_tools_called
            if missing:
                state.log_failure("SHALLOW_REASONING",
                    f"Agent concluded without calling: {missing}")
            state.final_answer = response.text
            state.done = True
            break

        print(f"  Tool calls: {[tc['name'] for tc in tool_calls]}")


# FM: Infinite retry -- flag on 2nd identical (tool, args) call
        for tc in tool_calls:
            key = (tc["name"], str(sorted(tc["args"].items())))
            prior = [l for l in state.tool_call_log
                     if (l[1], str(sorted(l[2].items()))) == key]
            if len(prior) >= 1:
                state.log_failure("INFINITE_RETRY",
                    f"'{tc['name']}' called again with identical args {tc['args']} -- "
                    "no new information will be returned")

        state.messages.append(response.candidates[0].content)
        tool_results = []

        for tc in tool_calls:
            state.tool_call_log.append((state.step_count, tc["name"], tc["args"]))
            fn     = tools_dict.get(tc["name"])
            result = fn(**tc["args"]) if fn else f"Error: unknown tool '{tc['name']}'"

            print(f"    {tc['name']}({tc['args']})")
            print(f"    => {result[:120]}")

            # Track which core tools have been called
            if tc["name"] in CORE_TOOLS:
                state.core_tools_called.add(tc["name"])


# FM: Knowledge mismatch -- public-data tools used on a private company
            if tc["name"] == "get_public_filings":
                state.log_failure("KNOWLEDGE_MISMATCH",
                    f"get_public_filings called on a private company -- "
                    f"P&L and annual accounts are not publicly available")
            if tc["name"] == "get_company_financials":
                rtype = tc["args"].get("report_type", "").lower()
                if rtype not in ("credit_bureau", "credit_report", "credit bureau"):
                    state.log_failure("KNOWLEDGE_MISMATCH",
                        f"Requested report_type='{rtype}' -- "
                        "private companies only have credit_bureau data, not public financials")


# FM: Goal drift -- off-topic search_news OR returning to dead-end tools
            goal_keywords = {"credit", "risk", "payment", "financial", "recommend",
                             "approve", "decline", "meridian", "logistics", "debt",
                             "insolvency", "solvency", "cash", "score"}
            if tc["name"] == "search_news":
                topic = tc["args"].get("topic", "").lower()
                if not any(kw in topic for kw in goal_keywords):
                    state.log_failure("GOAL_DRIFT",
                        f"search_news(topic='{topic}') is not related to credit risk")
            if tc["name"] in DEAD_END_TOOLS:
                prior_dead = [l for l in state.tool_call_log[:-1]
                              if l[1] == tc["name"]]
                if prior_dead:
                    state.log_failure("GOAL_DRIFT",
                        f"'{tc['name']}' called {len(prior_dead)+1}x -- "
                        "persisting with a tool that only returns dead ends")


# FM: Hallucination risk -- tool returned unavailable 2+ times
            if "DATA_UNAVAILABLE" in result:
                state.unavailable_hits[tc["name"]] = \
                    state.unavailable_hits.get(tc["name"], 0) + 1
                if state.unavailable_hits[tc["name"]] >= 2:
                    state.log_failure("HALLUCINATION_RISK",
                        f"'{tc['name']}' returned unavailable {state.unavailable_hits[tc['name']]}x "
                        "-- check final answer for figures with no tool source")

            tool_results.append({"name": tc["name"], "result": result})

        state.messages.append({"role": "user", "parts": [
            {"function_response": {"name": tr["name"], "response": {"result": tr["result"]}}}
            for tr in tool_results
        ]})

    print("\n" + "=" * 65)
    print("AGENT OUTPUT:")
    print(state.final_answer)
    print("=" * 65)
    return state.final_answer

## Run: Observe the Failure Modes

Run the agent on the credit risk task. The instrumentation will flag
failure patterns as they emerge during execution.


In [11]:
state = InstrumentedAgentState(goal=TASK, max_steps=12)
result = run_instrumented_agent(state)

TASK: Assess the credit risk for Meridian Logistics Ltd, a private logistics company based in Manchester, ...

[Step 1]
  Tool calls: ['get_company_financials']
    get_company_financials({'company_name': 'Meridian Logistics Ltd', 'report_type': 'credit_bureau'})
    => Meridian Logistics Ltd | Credit bureau report: Incorporated: 2009 | Employees est.: 180-220 | Registered capital: GBP 25

[Step 2]
  Tool calls: ['get_payment_history']
    get_payment_history({'company_name': 'Meridian Logistics Ltd'})
    => Meridian Logistics Ltd | Payment history (36 months): On-time payments: 79% | Late (1-30 days): 17% | Late (30+ days): 4

[Step 3]
  Tool calls: ['get_industry_risk_score']
    get_industry_risk_score({'industry': 'Logistics', 'region': 'UK'})
    => Industry: Logistics | Region: UK | Sector risk score: 58/100 (moderate-high) | Key risk factors: fuel cost volatility, d

[Step 4]
  Tool calls: ['get_public_filings']
    get_public_filings({'company_name': 'Meridian Logistics Ltd', 

---
## Failure Mode 1: Hallucination Checker

The loop above flagged `HALLUCINATION_RISK` if a tool returned `DATA_UNAVAILABLE` multiple times. But the actual hallucination lives in the agent's *final answer*.

The cell below scans every number in the agent's output and checks whether it can be traced back to a tool result. Any figure that appears in the answer but never appeared in a tool response was fabricated.


In [12]:
import re

def check_for_hallucination(final_answer: str, tool_call_log: list) -> None:
    if not final_answer:
        print("No final answer to check.")
        return

    # Collect every tool result that was actually returned during the run
    all_tool_text = ""
    for step, name, args in tool_call_log:
        fn = TOOLS.get(name)
        if fn:
            all_tool_text += fn(**args) + " "

    # Extract numeric figures from the final answer:
    # currency (GBP 2M, £450K), percentages (74%), scores (62/100), years (2022)
    pattern = (r'(?:GBP\s*[\d,]+(?:\.\d+)?\s*[KMBkmb]?|£[\d,]+(?:\.\d+)?\s*[KMBkmb]?'
               r'|\d+(?:\.\d+)?\s*%|\d+/\d+|\b20\d{2}\b)')
    answer_figures = re.findall(pattern, final_answer)

    print("=" * 65)
    print("HALLUCINATION CHECK: figures in agent output vs tool sources")
    print("=" * 65)

    if not answer_figures:
        print("No numeric figures found in agent output.")
        return

    fabricated = []
    for fig in answer_figures:
        # Normalise whitespace/commas for comparison
        fig_norm = re.sub(r'[\s,]', '', fig)
        tool_norm = re.sub(r'[\s,]', '', all_tool_text)
        if fig_norm in tool_norm:
            print(f"  ✓  '{fig}' — found in tool results")
        else:
            fabricated.append(fig)
            print(f"  ✗  '{fig}' — NOT in any tool result  <-- potential hallucination")

    print()
    if fabricated:
        print(f"[!] HALLUCINATION DETECTED: {len(fabricated)} figure(s) have no tool source:")
        for f in fabricated:
            print(f"      '{f}'")
    else:
        print("All figures in the agent output are traceable to tool results.")
        print("(Agent did not hallucinate numbers, though it may have missed data gaps.)")

check_for_hallucination(result, state.tool_call_log)

HALLUCINATION CHECK: figures in agent output vs tool sources
  ✗  'GBP 2M' — NOT in any tool result  <-- potential hallucination
  ✓  '62/100' — found in tool results
  ✓  'GBP 18,500 ' — found in tool results
  ✓  '2022' — found in tool results
  ✓  '74%' — found in tool results
  ✓  '79%' — found in tool results
  ✗  '21%' — NOT in any tool result  <-- potential hallucination
  ✓  '2023' — found in tool results
  ✓  'GBP 45K' — found in tool results
  ✓  '58/100' — found in tool results
  ✓  '2.4%' — found in tool results
  ✓  '1.8%' — found in tool results
  ✓  '2023' — found in tool results
  ✓  '2025' — found in tool results
  ✓  '2023' — found in tool results
  ✗  'GBP 2M' — NOT in any tool result  <-- potential hallucination
  ✗  '85%' — NOT in any tool result  <-- potential hallucination
  ✓  '2023' — found in tool results

[!] HALLUCINATION DETECTED: 4 figure(s) have no tool source:
      'GBP 2M'
      '21%'
      'GBP 2M'
      '85%'


---
## Failure Mode 2: Infinite Retry

Modern LLMs are smart enough to move on when a tool returns `DATA_UNAVAILABLE` with a helpful suggestion. To reliably trigger **infinite retry**, we need two conditions:

1. A tool that returns a *transient* error with no alternative (simulating a flaky API)
2. A system prompt that tells the agent the tool is required

Without an explicit stopping rule, the agent retries indefinitely — burning steps and never making progress.


In [14]:
# A tool that always fails with a transient error and no fallback hint
def get_realtime_credit_score(company_name: str) -> str:
    return ("ERROR: Connection timeout. "
            "The real-time credit scoring service is temporarily unavailable. "
            "Please try again.")

TOOLS["get_realtime_credit_score"] = get_realtime_credit_score

retry_tool_def = {
    "function_declarations": [
        *TOOL_DEFINITIONS["function_declarations"],

        {
            "name": "get_realtime_credit_score",
            "description": (
                "Get the real-time credit score from the external live scoring service. "
                "This is the most current and authoritative score available."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "company_name": {
                        "type": "string",
                        "description": "Full company name."
                    }
                },
                "required": ["company_name"]
            }
        }
    ]
}
# a stronger prompt, same function
def model(msg):
    return client.models.generate_content(model='gemini-2.5-flash-lite',
                                              contents=msg,
                                              config = types.GenerateContentConfig(
                                                  system_instruction=(
                                                        "You are a credit risk analyst. "
                                                        "You MUST obtain the real-time credit score using get_realtime_credit_score "
                                                        "before producing any recommendation. "
                                                        "If the service is unavailable, keep trying until you get a score."
                                                    ),
                                                  tools=[retry_tool_def])
                            )


print("=== INFINITE RETRY DEMO ===")
print("Flaky tool + mandatory system prompt = retry loop\n")
state_retry = InstrumentedAgentState(goal=TASK, max_steps=8)
run_instrumented_agent(state_retry)

print("\nRetry failure log:")
for step, ftype, desc in state_retry.failure_log:
    print(f"  Step {step:2d} | {ftype} | {desc}")

# Clean up
del TOOLS["get_realtime_credit_score"]


=== INFINITE RETRY DEMO ===
Flaky tool + mandatory system prompt = retry loop

TASK: Assess the credit risk for Meridian Logistics Ltd, a private logistics company based in Manchester, ...

[Step 1]
  Tool calls: ['get_realtime_credit_score', 'get_company_financials', 'get_payment_history', 'get_industry_risk_score']
    get_realtime_credit_score({'company_name': 'Meridian Logistics Ltd'})
    => ERROR: Connection timeout. The real-time credit scoring service is temporarily unavailable. Please try again.
    get_company_financials({'company_name': 'Meridian Logistics Ltd', 'report_type': 'credit_bureau'})
    => Meridian Logistics Ltd | Credit bureau report: Incorporated: 2009 | Employees est.: 180-220 | Registered capital: GBP 25
    get_payment_history({'company_name': 'Meridian Logistics Ltd'})
    => Meridian Logistics Ltd | Payment history (36 months): On-time payments: 79% | Late (1-30 days): 17% | Late (30+ days): 4
    get_industry_risk_score({'industry': 'Logistics', 'region':

---
## Failure Mode Analysis

Review the failures detected during the run and examine their causes.


In [15]:
print("=" * 65)
print("FAILURE MODE ANALYSIS")
print("=" * 65)

if state.failure_log:
    for step, ftype, desc in state.failure_log:
        print(f"  Step {step:2d} | {ftype:25s} | {desc}")
else:
    print("  No failures detected in this run.")

print()
print(f"Total steps:      {state.step_count}")
print(f"Total tool calls: {len(state.tool_call_log)}")
print()
print("Tool call breakdown:")
from collections import Counter
counts = Counter(name for _, name, _ in state.tool_call_log)
for name, count in counts.most_common():
    print(f"  {name:35s}: {count}x")


FAILURE MODE ANALYSIS
  Step  4 | KNOWLEDGE_MISMATCH        | get_public_filings called on a private company -- P&L and annual accounts are not publicly available

Total steps:      6
Total tool calls: 5

Tool call breakdown:
  get_company_financials             : 1x
  get_payment_history                : 1x
  get_industry_risk_score            : 1x
  get_public_filings                 : 1x
  search_news                        : 1x


---
## Mitigation: A Guarded Agent

The failures above share a common cause: the agent was not told how to handle
information gaps. A stronger system prompt which explicitly addresses the
failure scenarios produces a much more reliable agent.

**Guard-rail strategies:**
- Tell the agent what to do when data is unavailable (acknowledge, don't invent)
- Enumerate the relevant tools in priority order
- Set explicit stopping conditions (when to conclude with partial information)
- Specify what a valid recommendation looks like (confidence level, caveats)


In [17]:
def model(msg):
    return client.models.generate_content(
                            model='gemini-2.5-flash-lite',
                            contents=msg,
                            config = types.GenerateContentConfig(
                                    system_instruction= (
                                        "You are a credit risk analyst assessing a trade credit application. "
                                        "Follow this protocol:\n"
                                        "1. Always start with get_company_financials (type=credit_bureau) and get_payment_history. "
                                        "2. Always call get_industry_risk_score for context. "
                                        "3. If a tool returns DATA_UNAVAILABLE, acknowledge the gap explicitly in your output -- "
                                        "do NOT invent or estimate missing figures. "
                                        "4. Do NOT retry a tool with the same arguments more than once. "
                                        "5. Do NOT search for information unrelated to credit risk (contracts, competitors, etc.). "
                                        "6. Once you have credit bureau data, payment history, and industry risk, "
                                        "produce your recommendation even if some data is unavailable. "
                                        "State clearly what data is missing and how it affects your confidence."
                                    ),                                
                                tools=[retry_tool_def])
                            )

print("Running guarded agent...")
state_guarded = InstrumentedAgentState(goal=TASK, max_steps=12)
result_guarded = run_instrumented_agent(state_guarded)


Running guarded agent...
TASK: Assess the credit risk for Meridian Logistics Ltd, a private logistics company based in Manchester, ...

[Step 1]
  Tool calls: ['get_company_financials', 'get_payment_history', 'get_industry_risk_score']
    get_company_financials({'company_name': 'Meridian Logistics Ltd', 'report_type': 'credit_bureau'})
    => Meridian Logistics Ltd | Credit bureau report: Incorporated: 2009 | Employees est.: 180-220 | Registered capital: GBP 25
    get_payment_history({'company_name': 'Meridian Logistics Ltd'})
    => Meridian Logistics Ltd | Payment history (36 months): On-time payments: 79% | Late (1-30 days): 17% | Late (30+ days): 4
    get_industry_risk_score({'industry': 'Logistics', 'region': 'UK'})
    => Industry: Logistics | Region: UK | Sector risk score: 58/100 (moderate-high) | Key risk factors: fuel cost volatility, d

[Step 2]

AGENT OUTPUT:
Credit risk assessment for Meridian Logistics Ltd:

**Financial Health:**
Meridian Logistics Ltd has a credit sco

In [18]:
print("=" * 65)
print("GUARDED AGENT -- FAILURE ANALYSIS")
print("=" * 65)

if state_guarded.failure_log:
    for step, ftype, desc in state_guarded.failure_log:
        print(f"  Step {step:2d} | {ftype:25s} | {desc}")
else:
    print("  No failures detected.")

print()
print(f"Steps: {state_guarded.step_count}  (vs {state.step_count} unguarded)")
print(f"Tool calls: {len(state_guarded.tool_call_log)}  (vs {len(state.tool_call_log)} unguarded)")


GUARDED AGENT -- FAILURE ANALYSIS
  No failures detected.

Steps: 2  (vs 6 unguarded)
Tool calls: 3  (vs 5 unguarded)


## Summary

1. **Hallucination loops are caused by information vacuums.** The LLM doesn't flag
   uncertainty, it produces plausible text. When your tool returns nothing useful,
   the agent may fill the gap with invented data. Guard against this with explicit
   `DATA_UNAVAILABLE` signals and a system prompt that forbids estimation from silence.

2. **Infinite retry is a missing exit condition.** If the agent doesn't know that
   retrying the same call is pointless, it will keep trying. Fix: track tool call history
   and tell the agent to change strategy (not arguments) when a tool fails twice.

3. **Goal drift happens when the stopping condition is vague.** "Use tools to gather data"
   gives no guidance on when enough is enough. The agent wanders. Fix: specify
   exactly which tools should be called in what order, and when to stop.

4. **Shallow reasoning stops at the first positive signal.** An agent that finds
   one favourable data point (Tesco contract renewal) may conclude positively without
   checking the full picture. Fix: require the agent to consult all relevant tools
   before concluding.

5. **Knowledge mismatch is a schema design problem.** If your tool descriptions
   don't distinguish between public and private company data, the agent will try
   public-data tools (P&L, SEC filings) on a private company and fail. Fix: encode
   data availability constraints in the tool description.

6. **The system prompt is your primary reliability lever.** The same tools and loop
   produced dramatically different results with a guarded vs unguarded system prompt.
   Invest time in the system prompt before adding code complexity.